# CBO — Run the AI model on Google Colab

This notebook runs the local model (Ollama + Qwen2.5) on Colab's free GPU and exposes it at a public URL. Paste that URL into your app's `backend/.env` as `OLLAMA_BASE_URL`, and the app uses Colab's GPU instead of your machine.

**Steps:** Runtime → Change runtime type → GPU. Then run the cells top to bottom.

**Note:** Colab stops the model when the tab closes or after idle time. That's fine — the app automatically falls back to its built-in advisor when the model is unreachable, so nothing breaks. Just re-run this notebook and paste the new URL to bring the model back.

In [ ]:
# 1) Install and start Ollama, then pull the model
import os, subprocess, time
os.environ['OLLAMA_HOST'] = '0.0.0.0'
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(['ollama', 'serve'])
time.sleep(6)
!ollama pull qwen2.5:7b
print('\nModel ready.')

In [ ]:
# 2) Open a public tunnel to the model and print the URL
import subprocess, re
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
proc = subprocess.Popen(['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[-\w.]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('\n\n==============================================')
print('PUBLIC MODEL URL:', url)
print('Put this line in backend/.env, then restart the backend:')
print('OLLAMA_BASE_URL=' + (url or '<url-above>'))
print('==============================================')

In [ ]:
# 3) (Optional) Keep this cell running to hold the session open a bit longer
import time
print('Tunnel is live. Keep this tab open. Ctrl+C / stop to end.')
while True:
    time.sleep(60)